# Notas — Aula 11: Descriptors

Marco: os dois `@property` de `x` e `y` do `Robo` — quase idênticos desde a D2·Aula 2,
cada um validando "está dentro da grade?" com o mesmo código copiado e colado —
colapsam em um único descriptor reutilizável, `Coordenada`. `bateria`, que usa uma
regra diferente (`clamp` em vez de `raise`), ganha o mesmo tratamento com um segundo
descriptor, `Percentual`, provando que o mecanismo serve para qualquer atributo
validado, não só coordenada.

In [1]:
LADO_GRADE = 10

## `__dict__`: onde os atributos moram de verdade

Todo objeto guarda seus atributos de instância num dicionário próprio, acessível por
`objeto.__dict__`. `self.nome = nome` dentro de `__init__` não é mágica — é
literalmente inserir uma chave nesse dicionário. No robô, é aí que `x`, `y` e
`bateria` moram de verdade.

In [2]:
class Pessoa:
    def __init__(self, nome, idade):
        self.nome = nome
        self.idade = idade


p = Pessoa("Ana", 30)
print(p.__dict__)

{'nome': 'Ana', 'idade': 30}


## Atributo de classe vs. atributo de instância

Um atributo declarado direto no corpo da classe (fora de qualquer método) pertence à
**classe**, não a cada objeto — é o mesmo padrão de `Robo.LADO_GRADE`. Ele nunca
aparece no `__dict__` da instância; Python só sobe para o `__dict__` da classe quando
não encontra o nome na instância.

In [3]:
class Conta:
    banco = "Banco X"           # atributo de CLASSE

    def __init__(self, saldo):
        self.saldo = saldo      # atributo de INSTÂNCIA


c = Conta(100)
print(c.__dict__)
print(Conta.__dict__["banco"])

{'saldo': 100}
Banco X


### Sua vez

Complete `Robo`: `LADO_GRADE = 10` já é atributo de classe (não mude). No `__init__`,
atribua `self.nome = nome` e `self.bateria = 100` como atributos de instância.
Depois, confirme que `robo1.__dict__` tem `nome`/`bateria`, mas **não** tem
`LADO_GRADE`.

*Dica: só duas linhas dentro de `__init__`, no mesmo padrão de `Conta.saldo`.*

In [4]:
class Robo:
    LADO_GRADE = 10

    def __init__(self, nome):
        # TODO: atribua self.nome = nome e self.bateria = 100
        ...


robo1 = Robo("Wall-E")
print(robo1.__dict__)
print("LADO_GRADE" in robo1.__dict__)

{}
False


## Descriptor básico: um objeto que intercepta leitura e escrita

Em vez de um valor comum, a classe pode guardar um **objeto** que sabe reagir quando
alguém lê ou escreve o atributo — um *descriptor*. `__get__(self, instance, owner)`
roda na leitura; `__set__(self, instance, valor)` roda na escrita. O descriptor mora
**uma vez** em `Pessoa.__dict__`, compartilhado por todas as instâncias; quem muda é o
valor guardado em `instance.__dict__`.

In [5]:
class Atributo:
    def __init__(self, nome):
        self.nome = nome

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.nome]

    def __set__(self, instance, valor):
        instance.__dict__[self.nome] = valor


class Pessoa:
    nome = Atributo("nome")
    idade = Atributo("idade")

    def __init__(self, nome, idade):
        self.nome = nome
        self.idade = idade


p = Pessoa("Ana", 30)
print(p.nome, p.idade)

Ana 30


### Erro clássico: `__set__` chamando a si mesmo

Escrever `__set__` com `setattr(instance, self.nome, valor)` (em vez de mexer direto
em `instance.__dict__`) parece mais natural, mas `setattr(instance, self.nome, valor)`
é **exatamente** `instance.nome = valor` — e como `nome` continua sendo o descriptor,
isso chama `__set__` de novo, e de novo, até estourar a pilha. Regra: dentro de
`__get__`/`__set__`, sempre mexer em `instance.__dict__[...]` direto, nunca em
`instance.atributo`.

In [6]:
class AtributoQuebrado:
    def __init__(self, nome):
        self.nome = nome

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.nome)

    def __set__(self, instance, valor):
        setattr(instance, self.nome, valor)     # parece certo — não é


class PessoaQuebrada:
    nome = AtributoQuebrado("nome")

    def __init__(self, nome):
        self.nome = nome


try:
    PessoaQuebrada("Ana")
except RecursionError:
    print("RecursionError: maximum recursion depth exceeded")

RecursionError: maximum recursion depth exceeded


## `NaoNegativo`: o descriptor ganha uma regra de validação

`Sensor.alcance` nunca teve validação nenhuma — um `alcance` negativo sempre passou
batido. Um descriptor com `__set__` que recusa valores inválidos resolve isso **uma
vez**, reutilizável em qualquer atributo que precise da mesma regra.

In [7]:
class NaoNegativo:
    def __set_name__(self, owner, name):
        self.nome = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.nome]

    def __set__(self, instance, valor):
        if valor < 0:
            raise ValueError(f"{self.nome[1:]} não pode ser negativo, recebi {valor}")
        instance.__dict__[self.nome] = valor


class Sensor:
    alcance = NaoNegativo()

    def __init__(self, alcance=1):
        self.alcance = alcance


s1 = Sensor(3)
print(s1.alcance)
try:
    s1.alcance = -1
except ValueError as erro:
    print(f"{type(erro).__name__}: {erro}")

3
ValueError: alcance não pode ser negativo, recebi -1


Note que `NaoNegativo()` não recebe mais o nome como string — `__set_name__(self,
owner, name)` é chamado automaticamente quando a classe termina de ser definida, e o
descriptor descobre o próprio nome sozinho (`"alcance"`). Isso elimina o risco de
digitar `NaoNegativo("alcancee")` por engano, com um "e" a mais que ninguém percebe.

### Sua vez

Complete `TextoNaoVazio`: um descriptor que recusa string vazia ou só espaços em
branco, levantando `ValueError("nome não pode ser vazio")`; caso contrário, guarda
normalmente. Ele já está sendo usado como `Robo.nome` abaixo.

*Dica: `not valor.strip()` é `True` para `""` e para `"   "`; para guardar, use
`instance.__dict__[self.nome] = valor`, igual a `NaoNegativo`.*

In [8]:
class TextoNaoVazio:
    def __set_name__(self, owner, name):
        self.nome = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.nome]

    def __set__(self, instance, valor):
        # TODO: se valor for string vazia ou só espaços, levante
        # ValueError("nome não pode ser vazio"); senão, guarde normalmente
        ...


class Robo:
    nome = TextoNaoVazio()

    def __init__(self, nome):
        self.nome = nome


try:
    Robo("   ")
except ValueError as erro:
    print(f"{type(erro).__name__}: {erro}")

try:
    print(Robo("Wall-E").nome)
except KeyError:
    print("Complete o TODO acima para ver o resultado.")

Complete o TODO acima para ver o resultado.


## `property` também é um descriptor

`@property`, usado desde a D2·Aula 2, **é** um descriptor — o Python só embrulha
`__get__`/`__set__` numa sintaxe mais bonita. A diferença é de propósito: `property`
nasce para **um** atributo de **uma** classe; um descriptor escrito à mão, como
`NaoNegativo`, nasce para ser reaproveitado em vários atributos e várias classes.

In [9]:
class PessoaProperty:
    def __init__(self, nome):
        self._nome = nome

    @property
    def nome(self):
        return self._nome

    @nome.setter
    def nome(self, valor):
        if not valor:
            raise ValueError("nome não pode ser vazio")
        self._nome = valor


print(type(PessoaProperty.nome))
print(hasattr(PessoaProperty.nome, "__get__"), hasattr(PessoaProperty.nome, "__set__"))

<class 'property'>
True True


## `Coordenada`: adaptando `NaoNegativo` para a grade do robô

A regra de `x`/`y` do robô não é "não negativo" — é "dentro de um intervalo":
`0 <= valor < LADO_GRADE`. Em vez de fixar os limites no código do descriptor,
`Coordenada` recebe `minimo`/`maximo` no `__init__` — o mesmo descriptor serve para
qualquer grade, não só uma de lado 10.

In [10]:
class Coordenada:
    def __init__(self, minimo, maximo):
        self.minimo = minimo
        self.maximo = maximo

    def __set_name__(self, owner, name):
        self.nome_publico = name
        self.nome = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.nome]

    def __set__(self, instance, valor):
        if not (self.minimo <= valor < self.maximo):
            raise ValueError(
                f"{self.nome_publico}={valor} sai da grade "
                f"({self.minimo} a {self.maximo - 1})"
            )
        instance.__dict__[self.nome] = valor


class Teste:
    x = Coordenada(0, LADO_GRADE)

    def __init__(self, x):
        self.x = x


t = Teste(3)
print(t.x)
try:
    t.x = 99
except ValueError as erro:
    print(f"{type(erro).__name__}: {erro}")

3
ValueError: x=99 sai da grade (0 a 9)


## `x`/`y` do robô: de dezoito linhas para duas

Desde a D2·Aula 2, `Robo.x` e `Robo.y` eram dois `@property`/`@x.setter` quase
idênticos — nove linhas cada um, a mesma regra copiada e colada. Com `Coordenada`, os
dois viram uma linha cada.

In [11]:
class Robo:
    LADO_GRADE = 10
    x = Coordenada(0, LADO_GRADE)
    y = Coordenada(0, LADO_GRADE)

    def __init__(self, nome, x=0, y=0):
        self.nome = nome
        self.x = x
        self.y = y


robo1 = Robo("Wall-E")
print(robo1.x, robo1.y)
robo1.x = 3
print(robo1.x)
try:
    robo1.x = 99
except ValueError as erro:
    print(f"{type(erro).__name__}: {erro}")

0 0
3
ValueError: x=99 sai da grade (0 a 9)


### `Robo.x` sem nenhuma instância

`Robo.x` — perguntado pela **classe**, não por um robô — não é a coordenada de
ninguém. É o próprio objeto `Coordenada`, porque `__get__` tem exatamente esse desvio:
`if instance is None: return self`. Sem esse `if`, `Robo.x` quebraria tentando
`instance.__dict__[...]` com `instance` valendo `None`.

In [12]:
print(Robo.x)

## `Percentual`: um segundo descriptor, outra regra

`Robo.bateria` nunca usa `raise` — usa `clamp` (`max(0, min(100, valor))`), sem erro
nenhum, só "encosta" o valor no limite. O mesmo protocolo de descriptor serve para
essa regra completamente diferente: `__get__`/`__set__` são só um contrato de
interceptação; o que cada descriptor faz com isso é decisão dele.

In [13]:
class Percentual:
    def __set_name__(self, owner, name):
        self.nome = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.nome]

    def __set__(self, instance, valor):
        instance.__dict__[self.nome] = max(0, min(100, valor))


class Robo2:
    bateria = Percentual()

    def __init__(self, bateria=100):
        self.bateria = bateria


robo2 = Robo2(bateria=150)
print(robo2.bateria)
robo2.bateria = -30
print(robo2.bateria)

100
0


### Sua vez

`Sensor.confiabilidade` deve usar `Percentual` — mesma ideia de `Robo2.bateria`, outro
atributo. Complete o `__init__` de `Sensor2` atribuindo
`self.confiabilidade = confiabilidade`.

*Dica: uma linha, mesmo padrão de `self.alcance = alcance` logo acima.*

In [14]:
class Sensor2:
    alcance = NaoNegativo()
    confiabilidade = Percentual()

    def __init__(self, alcance=1, confiabilidade=100):
        self.alcance = alcance
        # TODO: atribua self.confiabilidade = confiabilidade
        ...


sensor = Sensor2(confiabilidade=140)
try:
    print(sensor.confiabilidade)
except KeyError:
    print("Complete o TODO acima para ver o resultado.")

Complete o TODO acima para ver o resultado.


## Para aprofundar

- Descriptor HowTo Guide (visão geral, com exemplos) — documentação oficial: https://docs.python.org/3/howto/descriptor.html
- Protocolo de descriptors (`__get__`, `__set__`, `__set_name__`) — referência oficial do modelo de dados: https://docs.python.org/3/reference/datamodel.html#implementing-descriptors
- `property()` (a classe embutida) — documentação oficial: https://docs.python.org/3/library/functions.html#property
- Descriptors na prática, com exemplos comentados — Real Python: https://realpython.com/python-descriptors/